# 📁 Working with Different File Formats in Python

Welcome! A **file format** is like a container: a water bottle, lunchbox, and suitcase all store something, but each is designed for different contents.

- **TXT** stores simple text.
- **CSV** stores rows and columns.
- **JSON** stores structured and nested information.
- **Excel** stores spreadsheet data across one or more sheets.

You have already learned file reading, file writing, pandas, and NumPy. Now you will combine those skills to load datasets, save cleaned data, exchange information between applications, and prepare data for AI and machine-learning projects.

---

## 🎯 Learning Objectives

By the end of this notebook, you will be able to:

- Recognize common data-file formats and select one for a task
- Create and read TXT, CSV, JSON, and Excel files
- Use Python's built-in modules and pandas
- Convert information from one format to another
- Handle paths safely with `pathlib`
- Use `with open(...)` correctly
- Handle common file-related errors
- Understand basic text encoding
- Compare formats by structure, readability, and use case

## 👀 File-Format Preview

| Format | Structure | Human-readable | Best use |
|---|---|---:|---|
| TXT | Plain text | Yes | Notes, logs, and reports |
| CSV | Rows and columns | Yes | Tabular datasets |
| JSON | Keys, values, and nesting | Yes | APIs and structured data |
| XLSX | Spreadsheet workbook | Partially | Reports and business data |

---

## 🧰 Installation and Imports

`pathlib` manages portable paths, `csv` and `json` handle their matching formats, and pandas works with table-shaped data. The `openpyxl` package enables `.xlsx` support.

In [ ]:
%pip install -q pandas openpyxl

In [ ]:
from pathlib import Path
import csv
import json
import pandas as pd

print("Libraries imported successfully.")
print("pandas version:", pd.__version__)

In [ ]:
# 🔵 Use a relative folder so the notebook works in Colab and locally.
data_directory = Path("sample_data")
data_directory.mkdir(exist_ok=True)

print("Working directory:", data_directory.resolve())

### ⚠️ Common File-Handling Mistakes

- Using an incorrect path or extension
- Forgetting to close an opened file
- Confusing a filename with the file's contents
- Assuming every format uses the same reading function

Prefer a **context manager**:

```python
with open(file_path, "r", encoding="utf-8") as file:
    content = file.read()
```

Python closes the file automatically when the `with` block finishes.

---

## 🧑‍🎓 Shared Student Dataset

We will store the same classroom records in several formats. Each dictionary represents one student, while its keys become DataFrame columns.

In [ ]:
student_records = [
    {"student_id": "BC2301", "name": "Ali", "subject": "Python", "marks": 82, "attendance": 91.5, "passed": True},
    {"student_id": "BC2302", "name": "Ayesha", "subject": "Python", "marks": 76, "attendance": 88.0, "passed": True},
    {"student_id": "BC2303", "name": "Hamza", "subject": "Python", "marks": 43, "attendance": 67.5, "passed": False},
]

students_df = pd.DataFrame(student_records)
students_df

---

## 📝 Working with TXT Files

A TXT file stores plain characters. It does not automatically understand rows, columns, or data types. TXT is useful for notes, logs, and simple reports when you control the layout.

In [ ]:
text_file_path = data_directory / "students.txt"

with open(text_file_path, "w", encoding="utf-8") as file:
    for student in student_records:
        line = (
            f"{student['student_id']} | {student['name']} | "
            f"{student['subject']} | {student['marks']}\n"
        )
        file.write(line)

print("Created:", text_file_path)

In [ ]:
with open(text_file_path, "r", encoding="utf-8") as file:
    all_text = file.read()

print(all_text)

In [ ]:
with open(text_file_path, "r", encoding="utf-8") as file:
    first_line = file.readline()

with open(text_file_path, "r", encoding="utf-8") as file:
    all_lines = file.readlines()

print("First line:", first_line.strip())
print("Number of lines:", len(all_lines))
print("Lines as a list:", all_lines)

### 🟡 What changed?

- `.read()` returns the complete content as one string.
- `.readline()` returns the next line.
- `.readlines()` returns a list of lines.

**Warning:** Reading moves the file cursor. Reopen the file or use `.seek(0)` before reading it again.

### 🧪 Guided Practice — Course Summary

Create `course_summary.txt` containing the course name, student count, and average marks. The starter cell remains executable; replace the placeholders to complete it.

In [ ]:
course_summary_path = data_directory / "course_summary.txt"

# 🟢 Replace these placeholders with your answers.
course_name = "TODO"
student_count = None
average_marks = None

if None not in (student_count, average_marks):
    with open(course_summary_path, "w", encoding="utf-8") as file:
        file.write(f"Course: {course_name}\n")
        file.write(f"Students: {student_count}\n")
        file.write(f"Average marks: {average_marks:.2f}\n")
    print(course_summary_path.read_text(encoding="utf-8"))
else:
    print("Complete the placeholders, then run this cell again.")

<details>
<summary>💡 Hint — click to reveal</summary>

Use `len(students_df)` and `students_df['marks'].mean()`.

</details>

<details>
<summary>✅ Solution — click to reveal</summary>

```python
course_name = "Python"
student_count = len(students_df)
average_marks = students_df["marks"].mean()

with open(course_summary_path, "w", encoding="utf-8") as file:
    file.write(f"Course: {course_name}\n")
    file.write(f"Students: {student_count}\n")
    file.write(f"Average marks: {average_marks:.2f}\n")
```

</details>

**When should I use TXT?** Use it for simple human-readable notes, logs, or reports that do not require an automatic table structure.

---

## 📊 Working with CSV Files

**CSV** means comma-separated values. Each line normally represents one row, and commas separate columns. CSV is widely used for datasets, but it does not preserve spreadsheet formatting, formulas, or multiple sheets.

```text
student_id,name,marks
BC2301,Ali,82
BC2302,Ayesha,76
```

### Built-in `csv` Module

In [ ]:
csv_builtin_path = data_directory / "students_builtin.csv"
fieldnames = list(student_records[0].keys())

with open(csv_builtin_path, "w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(student_records)

with open(csv_builtin_path, "r", newline="", encoding="utf-8") as file:
    loaded_rows = list(csv.DictReader(file))

print("First loaded row:", loaded_rows[0])

`newline=""` prevents unwanted blank lines on some operating systems. `DictReader` loads CSV values as strings, so you may need to convert their data types.

### pandas Approach

In [ ]:
csv_file_path = data_directory / "students.csv"
students_df.to_csv(csv_file_path, index=False)

loaded_csv_df = pd.read_csv(csv_file_path)
loaded_csv_df

In [ ]:
print("Shape:", loaded_csv_df.shape)
print("Columns:", loaded_csv_df.columns.tolist())
print("Data types:\n", loaded_csv_df.dtypes)
loaded_csv_df.head()

`index=False` prevents pandas from saving the DataFrame's row index as an unnecessary extra column.

### 🧪 Guided Practice — Passed Students

Filter students who passed, save them as `passed_students.csv`, and load the file again.

In [ ]:
passed_csv_path = data_directory / "passed_students.csv"

# 🟢 Replace None with the required expressions.
passed_students_df = None

if passed_students_df is not None:
    passed_students_df.to_csv(passed_csv_path, index=False)
    reloaded_passed_df = pd.read_csv(passed_csv_path)
    display(reloaded_passed_df)
else:
    print("Complete the filter, then run this cell again.")

<details>
<summary>💡 Hint — click to reveal</summary>

Filter rows with `students_df[students_df['passed']]`.

</details>

<details>
<summary>✅ Solution — click to reveal</summary>

```python
passed_students_df = students_df[students_df["passed"]]
passed_students_df.to_csv(passed_csv_path, index=False)
reloaded_passed_df = pd.read_csv(passed_csv_path)
display(reloaded_passed_df)
```

</details>

### ⚠️ CSV Mistakes

- Forgetting `index=False`
- Assuming every file uses commas
- Receiving numeric values as text
- Ignoring missing values or encoding

A **TSV** file follows the same table idea but uses tabs instead of commas.

In [ ]:
tsv_file_path = data_directory / "students.tsv"
students_df.to_csv(tsv_file_path, sep="\t", index=False)
tsv_df = pd.read_csv(tsv_file_path, sep="\t")
tsv_df

**When should I use CSV or TSV?** Use them for portable, table-shaped datasets without spreadsheet formatting.

---

## 🧩 Working with JSON Files

**JSON** means JavaScript Object Notation. It stores keys, values, lists, and nested objects, which makes it a common format for APIs and web applications.

| Python | JSON |
|---|---|
| `dict` | object |
| `list` | array |
| `str` | string |
| `int` / `float` | number |
| `True` / `False` | true / false |
| `None` | null |

In [ ]:
json_file_path = data_directory / "students.json"

with open(json_file_path, "w", encoding="utf-8") as file:
    json.dump(student_records, file, indent=4, ensure_ascii=False)

with open(json_file_path, "r", encoding="utf-8") as file:
    loaded_json_data = json.load(file)

print("Records loaded:", len(loaded_json_data))
print("First record:", loaded_json_data[0])
pd.DataFrame(loaded_json_data)

### Four Similar JSON Functions

- `json.dump(data, file)` writes Python data to a file.
- `json.dumps(data)` converts Python data to a JSON string.
- `json.load(file)` reads JSON from a file.
- `json.loads(text)` converts a JSON string to Python data.

In [ ]:
json_text = json.dumps(student_records[0], indent=2)
python_record = json.loads(json_text)

print(json_text)
print("Back in Python:", python_record)

### 🧪 Guided Practice — Course Information

Create `course_information.json` with a course name, instructor, total students, list of topics, and active status.

In [ ]:
course_json_path = data_directory / "course_information.json"

# 🟢 Replace None with a dictionary containing all required fields.
course_information = None

if course_information is not None:
    with open(course_json_path, "w", encoding="utf-8") as file:
        json.dump(course_information, file, indent=4, ensure_ascii=False)
    print(course_json_path.read_text(encoding="utf-8"))
else:
    print("Create the dictionary, then run this cell again.")

<details>
<summary>💡 Hint — click to reveal</summary>

Use strings for names, an integer for the count, a list for topics, and `True` for active status.

</details>

<details>
<summary>✅ Solution — click to reveal</summary>

```python
course_information = {
    "course_name": "Python for AI & Data Science",
    "instructor": "Saad",
    "total_students": len(students_df),
    "topics": ["Python", "pandas", "File Handling"],
    "active": True,
}

with open(course_json_path, "w", encoding="utf-8") as file:
    json.dump(course_information, file, indent=4, ensure_ascii=False)
```

</details>

### ⚠️ JSON Mistakes

- Manual JSON requires double quotes, not single quotes.
- JSON does not allow trailing commas.
- JSON uses `true`, while Python uses `True`.
- Some objects, such as a `Path` or DataFrame, require conversion before serialization.

**When should I use JSON?** Use it for APIs, configuration, or human-readable data with nested structures.

---

## 📗 Working with Excel Files

An Excel `.xlsx` workbook can contain multiple sheets, formatting, formulas, and charts. pandas focuses on reading and writing its table-shaped data, while `openpyxl` provides the Excel engine.

In [ ]:
excel_file_path = data_directory / "students.xlsx"

students_df.to_excel(excel_file_path, sheet_name="Students", index=False)
loaded_excel_df = pd.read_excel(excel_file_path, sheet_name="Students")
loaded_excel_df

### Multiple Sheets

We will create separate sheets for all, passed, failed, and summary records.

In [ ]:
passed_df = students_df[students_df["passed"]]
failed_df = students_df[~students_df["passed"]]
summary_df = pd.DataFrame({
    "metric": ["Total students", "Average marks", "Highest marks", "Lowest marks", "Pass count", "Fail count"],
    "value": [
        len(students_df), students_df["marks"].mean(), students_df["marks"].max(),
        students_df["marks"].min(), len(passed_df), len(failed_df)
    ],
})

multi_sheet_path = data_directory / "student_workbook.xlsx"
with pd.ExcelWriter(multi_sheet_path, engine="openpyxl") as writer:
    students_df.to_excel(writer, sheet_name="All Students", index=False)
    passed_df.to_excel(writer, sheet_name="Passed Students", index=False)
    failed_df.to_excel(writer, sheet_name="Failed Students", index=False)
    summary_df.to_excel(writer, sheet_name="Summary", index=False)

excel_file = pd.ExcelFile(multi_sheet_path)
print("Sheet names:", excel_file.sheet_names)

all_sheets = pd.read_excel(multi_sheet_path, sheet_name=None)
print("Loaded sheet keys:", list(all_sheets))
display(all_sheets["Summary"])

### 🧪 Guided Practice — Student Report

Create `student_report.xlsx` with `All Students`, `Passed`, and `Failed` sheets.

In [ ]:
student_report_path = data_directory / "student_report.xlsx"

# 🟢 Set this to True when you are ready to complete the task.
create_report = False

if create_report:
    # TODO: Use pd.ExcelWriter and write the three DataFrames.
    print("Add your ExcelWriter solution here.")
else:
    print("Review the hint, complete the writer block, and set create_report to True.")

<details>
<summary>💡 Hint — click to reveal</summary>

Use `with pd.ExcelWriter(student_report_path) as writer:` and call `.to_excel()` three times with different sheet names.

</details>

<details>
<summary>✅ Solution — click to reveal</summary>

```python
with pd.ExcelWriter(student_report_path, engine="openpyxl") as writer:
    students_df.to_excel(writer, sheet_name="All Students", index=False)
    passed_df.to_excel(writer, sheet_name="Passed", index=False)
    failed_df.to_excel(writer, sheet_name="Failed", index=False)
```

</details>

### ⚠️ Excel Mistakes

- Missing the `openpyxl` dependency
- Using an incorrect sheet name
- Forgetting `index=False`
- Trying to read CSV with `read_excel()`
- Overwriting an existing workbook unintentionally

**When should I use Excel?** Use it for business reports, formatted spreadsheets, or related tables across multiple sheets.

---

## 🔄 Converting Between File Formats

Conversion usually means: **read the source → store it in Python → write the destination → verify the result**.

In [ ]:
conversion_df = pd.read_csv(csv_file_path)

converted_json_path = data_directory / "students_from_csv.json"
converted_excel_path = data_directory / "students_from_csv.xlsx"

conversion_df.to_json(converted_json_path, orient="records", indent=4)
conversion_df.to_excel(converted_excel_path, index=False)

json_check_df = pd.read_json(converted_json_path)
excel_check_df = pd.read_excel(converted_excel_path)

print("CSV → JSON rows:", len(json_check_df))
print("CSV → Excel columns:", excel_check_df.columns.tolist())

`orient='records'` writes each DataFrame row as one JSON object inside a list.

In [ ]:
# JSON → CSV
json_source_df = pd.read_json(json_file_path)
json_to_csv_path = data_directory / "students_from_json.csv"
json_source_df.to_csv(json_to_csv_path, index=False)

# Excel → CSV
excel_source_df = pd.read_excel(excel_file_path)
excel_to_csv_path = data_directory / "students_from_excel.csv"
excel_source_df.to_csv(excel_to_csv_path, index=False)

print("JSON → CSV verified rows:", len(pd.read_csv(json_to_csv_path)))
print("Excel → CSV verified rows:", len(pd.read_csv(excel_to_csv_path)))

---

## 🛡️ Safe Paths and Common Errors

`Path` makes file paths readable and portable.

In [ ]:
print("Exists:", csv_file_path.exists())
print("Is a file:", csv_file_path.is_file())
print("Name:", csv_file_path.name)
print("Stem:", csv_file_path.stem)
print("Suffix:", csv_file_path.suffix)
print("Parent:", csv_file_path.parent)

In [ ]:
missing_file_path = data_directory / "missing_file.csv"

try:
    missing_df = pd.read_csv(missing_file_path)
except FileNotFoundError:
    print(f"File not found: {missing_file_path}")

Common exceptions include:

- `FileNotFoundError`: the path does not exist
- `PermissionError`: Python cannot access the location
- `UnicodeDecodeError`: the selected text encoding is incorrect
- `json.JSONDecodeError`: the JSON syntax is invalid
- `pd.errors.EmptyDataError`: a data file is empty

Check a path before reading when a missing file is expected. Use `try`/`except` when you need to respond gracefully to failure.

---

## 🔤 Encoding and Data Quality

**UTF-8** is a widely used character encoding that supports English, Urdu, and many other writing systems.

In [ ]:
multilingual_path = data_directory / "multilingual_names.txt"
multilingual_text = "Saad | سعد\nAyesha | عائشہ\n"

multilingual_path.write_text(multilingual_text, encoding="utf-8")
print(multilingual_path.read_text(encoding="utf-8"))

In [ ]:
print("Missing values by column:\n", students_df.isna().sum())
print("Column names:", students_df.columns.tolist())
print("Data types:\n", students_df.dtypes)

Before analysis, check for missing values, unexpected column names, extra spaces, incorrect separators, and unsuitable data types.

---

## ⚖️ File-Format Comparison

| Format | Structure | Human-readable | Nested data | Multiple sheets | Common use |
|---|---|---:|---:|---:|---|
| TXT | Plain characters | Yes | No | No | Notes and logs |
| CSV | Rows and columns | Yes | No | No | Portable datasets |
| TSV | Rows and tab-separated columns | Yes | No | No | Text containing commas |
| JSON | Keys, values, lists, objects | Yes | Yes | No | APIs and configuration |
| XLSX | Spreadsheet workbook | Partially | No | Yes | Reports and business data |
| Pickle | Python binary objects | No | Yes | No | Trusted Python-only storage |

> 🔴 **Security warning:** Never load a Pickle file from an unknown or untrusted source. A malicious Pickle file may execute harmful code.

---

# 🛠️ Mini Project: Student Data File Converter

Apply the complete workflow in three levels. Each starter remains safe to run before completion.

## Starter — DataFrame to CSV

Create at least four student records, save them as `student_project.csv`, reload the file, and display it.

In [ ]:
project_csv_path = data_directory / "student_project.csv"

# 🟢 Replace None with a DataFrame containing at least four students.
project_students_df = None

if project_students_df is not None:
    project_students_df.to_csv(project_csv_path, index=False)
    display(pd.read_csv(project_csv_path))
else:
    print("Create the project DataFrame, then run this cell again.")

<details>
<summary>💡 Starter hint</summary>

Create a list of dictionaries, pass it to `pd.DataFrame()`, and call `.to_csv(..., index=False)`.

</details>

<details>
<summary>✅ Starter solution</summary>

```python
project_students_df = pd.DataFrame([
    {"name": "Ali", "marks": 82},
    {"name": "Ayesha", "marks": 76},
    {"name": "Hamza", "marks": 43},
    {"name": "Sana", "marks": 91},
])
project_students_df.to_csv(project_csv_path, index=False)
display(pd.read_csv(project_csv_path))
```

</details>

## Intermediate — CSV to JSON

Load the project CSV, add a `passed` column using a mark of 50, save JSON, and verify the record count.

In [ ]:
project_json_path = data_directory / "student_project.json"

if project_csv_path.exists():
    intermediate_df = pd.read_csv(project_csv_path)
    # 🟢 TODO: Create the passed column and save JSON with orient="records".
    print("Project CSV found. Complete the two TODO operations.")
else:
    print("Complete the Starter level first.")

<details>
<summary>💡 Intermediate hint</summary>

Use `intermediate_df['passed'] = intermediate_df['marks'] >= 50`, then `.to_json(..., orient='records', indent=4)`.

</details>

<details>
<summary>✅ Intermediate solution</summary>

```python
intermediate_df = pd.read_csv(project_csv_path)
intermediate_df["passed"] = intermediate_df["marks"] >= 50
intermediate_df.to_json(project_json_path, orient="records", indent=4)
verified_records = pd.read_json(project_json_path)
print("Verified records:", len(verified_records))
```

</details>

## Bonus — Multi-Sheet Excel Report

Create all-student, passed, failed, and summary sheets, then display the workbook's sheet names.

In [ ]:
project_excel_path = data_directory / "student_project_report.xlsx"

# 🟢 Set to True after adding your solution.
build_bonus_report = False

if build_bonus_report:
    print("Add your multi-sheet ExcelWriter solution here.")
else:
    print("Complete the bonus solution and set build_bonus_report to True.")

<details>
<summary>💡 Bonus hint</summary>

Filter on the `passed` column, build a one-row summary DataFrame, and write each DataFrame with a different `sheet_name`.

</details>

<details>
<summary>✅ Bonus solution</summary>

```python
bonus_df = pd.read_json(project_json_path)
bonus_passed = bonus_df[bonus_df["passed"]]
bonus_failed = bonus_df[~bonus_df["passed"]]
bonus_summary = pd.DataFrame({
    "total": [len(bonus_df)],
    "passed": [len(bonus_passed)],
    "failed": [len(bonus_failed)],
    "average_marks": [bonus_df["marks"].mean()],
})

with pd.ExcelWriter(project_excel_path, engine="openpyxl") as writer:
    bonus_df.to_excel(writer, sheet_name="All Students", index=False)
    bonus_passed.to_excel(writer, sheet_name="Passed", index=False)
    bonus_failed.to_excel(writer, sheet_name="Failed", index=False)
    bonus_summary.to_excel(writer, sheet_name="Summary", index=False)

print(pd.ExcelFile(project_excel_path).sheet_names)
```

</details>

---

# 🧠 Self-Assessment Quiz

1. What is the main difference between CSV and JSON?
2. Why is `index=False` commonly used with `to_csv()` and `to_excel()`?
3. Which format is most suitable for nested API data?
4. Why should `with open(...)` be used?
5. How can Python check whether a file exists before reading it?

<details>
<summary>✅ Quiz answers — click after attempting</summary>

1. CSV represents rows and columns; JSON supports keys, lists, and nested objects.
2. It prevents pandas from saving its row index as an extra column.
3. JSON.
4. It closes the file automatically, even if an error occurs.
5. Use `Path.exists()` or `Path.is_file()`.

</details>

# ✅ Recap Table

| Function or syntax | Format | Purpose | Short example |
|---|---|---|---|
| `Path()` | All | Create a path | `Path("data.csv")` |
| `open()` | Text | Open a file | `open(path, "r")` |
| `.read()` | TXT | Read all text | `file.read()` |
| `.write()` | TXT | Write text | `file.write(text)` |
| `csv.DictReader` | CSV | Read rows as dictionaries | `csv.DictReader(file)` |
| `csv.DictWriter` | CSV | Write dictionary rows | `csv.DictWriter(...)` |
| `json.load()` | JSON | Read JSON file | `json.load(file)` |
| `json.dump()` | JSON | Write JSON file | `json.dump(data, file)` |
| `pd.read_csv()` | CSV | Load a table | `pd.read_csv(path)` |
| `.to_csv()` | CSV | Save a table | `df.to_csv(path)` |
| `pd.read_json()` | JSON | Load JSON table | `pd.read_json(path)` |
| `.to_json()` | JSON | Save JSON | `df.to_json(path)` |
| `pd.read_excel()` | XLSX | Load a sheet | `pd.read_excel(path)` |
| `.to_excel()` | XLSX | Save a sheet | `df.to_excel(path)` |
| `pd.ExcelWriter()` | XLSX | Write multiple sheets | `pd.ExcelWriter(path)` |
| `Path.exists()` | All | Check existence | `path.exists()` |

## 🧭 File-Format Decision Guide

| Requirement | Recommended format | Reason |
|---|---|---|
| Simple notes or logs | TXT | Minimal and easy to read |
| Machine-learning table | CSV | Portable rows and columns |
| API response | JSON | Supports structured and nested data |
| Business spreadsheet | XLSX | Works with Excel and formatting |
| Multiple spreadsheet tables | XLSX | Supports multiple sheets |
| Human-readable nested data | JSON | Preserves lists and objects |

---

## 🎉 Module Complete

You can now create, read, compare, and convert common file formats; select the right format for a task; and handle paths, encoding, and common errors safely.

Next, **Module 6 — APIs and Web Data Collection** will use these file-handling skills to collect and store information from online services.